In [1]:
# =========================================================
# Prequential GP evaluation — CORRECTED
#
# Changes vs original:
#   1. Pointwise scoring uses predict_y (latent var + noise),
#      instead of predict_f. Means are unchanged → MAE/RMSE
#      are identical; only variance-based scores change.
#   2. Joint scoring keeps predict_f(full_cov=True) for the
#      latent covariance, then adds sigma2_eps * I to the
#      DIAGONAL by hand (predict_y(full_cov=True) is buggy in
#      GPflow — it adds noise to every element, not just the
#      diagonal; see GPflow issue #1461). This both corrects
#      the score and regularises the near-singular matrices
#      that caused the multiple-kernel blow-ups.
#   3. multivariate_log_score now REPORTS when it falls back
#      to the marginal score (Sigma not PD) instead of doing
#      it silently. A 'fallback' flag is propagated.
#   4. Aggregation expanded:
#        - per-sim then across-sim summaries
#        - mean, median, sd, var, 95% percentile interval,
#          95% CI on the mean
#        - aggregate SUM (total prequential log score) per sim,
#          then summarised across sims (mean & median of totals)
#        - everything computed for ALL runs and for
#          SUCCESSFULLY-OPTIMISED runs separately
# =========================================================

In [2]:
import numpy as np
import pandas as pd

def rbf_kernel(t, lengthscale, variance):
    T_mat = t.reshape(-1, 1)
    sqdist = (T_mat - T_mat.T)**2
    return variance * np.exp(-0.5 * sqdist / lengthscale**2)

def sample_gp(cov, n_samples=1):
    jitter = 1e-6 * np.mean(np.diag(cov)) * np.eye(len(cov))
    L = np.linalg.cholesky(cov + jitter)
    return L @ np.random.randn(len(cov), n_samples)

def simulate_gp_scenarios(num_GPs=8, T=100, noise_var=0.05, seed=42):
    np.random.seed(seed)
    
    # NEW: Integers from 1 to T
    t = np.arange(1, T + 1)
    results = []

    # Since the time scale expanded from 10 to 100 (10x larger), 
    # we must multiply all lengthscales by 10 to keep the exact same smoothness!
    
    # Scenario 1: Exchangeable
    var_mu_1, ls_mu_1 = 2.0, 20.0  # lengthscale was 2.0 -> 20.0
    var_x_1, ls_x_1 = 1.0, 10.0    # lengthscale was 1.0 -> 10.0
    K_mu_1 = rbf_kernel(t, ls_mu_1, var_mu_1)
    K_x_1 = rbf_kernel(t, ls_x_1, var_x_1)
    mu_1 = sample_gp(K_mu_1).flatten()
    for i in range(num_GPs):
        D_i = sample_gp(K_x_1).flatten()
        x_i = mu_1 + D_i
        y_i = x_i + np.random.normal(0, np.sqrt(noise_var), T)
        results.append({"scenario": "1_exchangeable", "unit_id": i, "time": t, "y": y_i, "true_x": x_i, "true_mu": mu_1})

    # Scenario 2: Nearly independent
    var_mu_2, ls_mu_2 = 0.1, 20.0  # lengthscale was 2.0 -> 20.0
    var_x_2, ls_x_2 = 1.0, 10.0    # lengthscale was 1.0 -> 10.0
    K_mu_2 = rbf_kernel(t, ls_mu_2, var_mu_2)
    K_x_2 = rbf_kernel(t, ls_x_2, var_x_2)
    mu_2 = sample_gp(K_mu_2).flatten()
    for i in range(num_GPs):
        D_i = sample_gp(K_x_2).flatten()
        x_i = mu_2 + D_i
        y_i = x_i + np.random.normal(0, np.sqrt(noise_var), T)
        results.append({"scenario": "2_nearly_independent", "unit_id": i, "time": t, "y": y_i, "true_x": x_i, "true_mu": mu_2})

    # Scenario 3: Non-exchangeable IID
    for i in range(num_GPs):
        ls_i = 10.0  # Scaled by 10
        var_i = 2.0
        K_i = rbf_kernel(t, lengthscale=ls_i, variance=var_i)
        x_i = sample_gp(K_i).flatten()
        y_i = x_i + np.random.normal(0, np.sqrt(noise_var), T)
        results.append({"scenario": "3_non_exchangeable_iid", "unit_id": i, "time": t, "y": y_i, "true_x": x_i, "true_mu": np.zeros(T)})



    # Scenario 4: Non-exchangeable non IID
    for i in range(num_GPs):
        ls_i = (0.5 + (i * 0.5)) * 10.0  # Scaled by 10
        var_i = 0.5 + (i * 0.4)
        K_i = rbf_kernel(t, lengthscale=ls_i, variance=var_i)
        x_i = sample_gp(K_i).flatten()
        y_i = x_i + np.random.normal(0, np.sqrt(noise_var), T)
        results.append({"scenario": "4_non_exchangeable_non_iid", "unit_id": i, "time": t, "y": y_i, "true_x": x_i, "true_mu": np.zeros(T)})

    df_list = [pd.DataFrame(res) for res in results]
    return pd.concat(df_list, ignore_index=True)

df = simulate_gp_scenarios(num_GPs=8, T=100)
pd.set_option('display.max_rows', 200)
print(df.head(200).to_string())


# =========================================================
# RUN THE SIMULATION MULTIPLE TIMES
# =========================================================

num_sim = 50        # How many different simulation datasets you want
num_GPs = 12          # How many units/tasks per scenario
T = 112              # How many time points per GP

# We will store all the generated DataFrames in a dictionary
all_simulations = {} 

# Loop through and generate 'num_sim' different datasets
for sim_index in range(num_sim):
    
    # By passing 'sim_index' as the seed, every simulation gets a unique seed!
    # Run 0 uses seed=0, Run 1 uses seed=1, etc.
    sim_data = simulate_gp_scenarios(
        num_GPs=num_GPs, 
        T=T, 
        noise_var=1.0, 
        seed=sim_index
    )
    
    # Save the dataframe into the dictionary with a name like 'sim_0', 'sim_1', etc.
    all_simulations[f"sim_{sim_index}"] = sim_data

print(f"Successfully generated {num_sim} unique simulation datasets!")

# =========================================================
# HOW TO ACCESS A SPECIFIC SIMULATION
# =========================================================

# If you want to look at the 5th simulation dataset (index 4):
df_sim_4 = all_simulations["sim_4"]

print("\nShowing the first few rows of simulation #4:")
print(df_sim_4.head())

           scenario  unit_id  time         y    true_x   true_mu
0    1_exchangeable        0     1 -0.632908 -0.712911  0.702460
1    1_exchangeable        0     2 -0.633072 -0.758467  0.691807
2    1_exchangeable        0     3 -0.549730 -0.791908  0.683399
3    1_exchangeable        0     4 -0.574421 -0.810058  0.683368
4    1_exchangeable        0     5 -1.126444 -0.818387  0.684895
5    1_exchangeable        0     6 -1.022607 -0.812903  0.691070
6    1_exchangeable        0     7 -0.667470 -0.782635  0.706919
7    1_exchangeable        0     8 -0.621420 -0.736306  0.726961
8    1_exchangeable        0     9 -0.554820 -0.669988  0.750638
9    1_exchangeable        0    10  0.282728 -0.578769  0.782945
10   1_exchangeable        0    11 -0.346191 -0.473846  0.817902
11   1_exchangeable        0    12 -0.094530 -0.348450  0.857928
12   1_exchangeable        0    13  0.006247 -0.207074  0.903744
13   1_exchangeable        0    14  0.092807 -0.052849  0.947161
14   1_exchangeable      

In [3]:
all_simulations

{'sim_0':                         scenario  unit_id  time         y    true_x   true_mu
 0                 1_exchangeable        0     1  0.092725  1.633522  2.494748
 1                 1_exchangeable        0     2  1.916799  1.853537  2.519918
 2                 1_exchangeable        0     3  2.230418  2.073912  2.544647
 3                 1_exchangeable        0     4  2.534083  2.301902  2.577664
 4                 1_exchangeable        0     5  1.936440  2.533756  2.617826
 ...                          ...      ...   ...       ...       ...       ...
 5371  4_non_exchangeable_non_iid       11   108  4.834297  4.270022  0.000000
 5372  4_non_exchangeable_non_iid       11   109  4.279756  4.244307  0.000000
 5373  4_non_exchangeable_non_iid       11   110  3.139316  4.218242  0.000000
 5374  4_non_exchangeable_non_iid       11   111  4.302942  4.190421  0.000000
 5375  4_non_exchangeable_non_iid       11   112  3.952304  4.167602  0.000000
 
 [5376 rows x 6 columns],
 'sim_1':      

In [4]:
df.iloc[10:20, [1, 2, 3]]


,unit_id,time,y
10,0,11,-0.346191
11,0,12,-0.094530
12,0,13,0.006247
13,0,14,0.092807
14,0,15,0.037844
15,0,16,0.448913
16,0,17,0.277222
17,0,18,0.566654
18,0,19,0.675628
19,0,20,0.957427


In [5]:
# Save all simulated datasets into one Excel file
with pd.ExcelWriter("all_simulated_datasets_12_units_50_dfs.xlsx", engine="openpyxl") as writer:
    for sim_name, sim_df in all_simulations.items():
        # Write each dataframe to its own sheet named after the simulation (e.g., 'sim_0')
        sim_df.to_excel(writer, sheet_name=sim_name, index=False)

print("Simulated datasets saved to 'all_simulated_datasets.xlsx'")

Simulated datasets saved to 'all_simulated_datasets.xlsx'


In [6]:
import numpy as np
import pandas as pd
import tensorflow as tf
import gpflow
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


# ---------------------------------------------------------
# Brownian motion kernel
# ---------------------------------------------------------
class BrownianMotion(gpflow.kernels.Kernel):
    def __init__(self, variance=1.0):
        super().__init__()
        self.variance = gpflow.Parameter(variance, transform=gpflow.utilities.positive())

    def K(self, X, X2=None):
        if X2 is None:
            X2 = X
        t1 = X[:, 0:1]
        t2 = X2[:, 0:1]
        return self.variance * tf.minimum(t1, tf.transpose(t2))

    def K_diag(self, X):
        return self.variance * tf.reshape(X[:, 0], (-1,))


# ---------------------------------------------------------
# Heteroscedastic noise helper (kept for future use)
# ---------------------------------------------------------
class UnitVariance(gpflow.functions.Function):
    def __init__(self, n_units, active_task, init_var=0.05, lower=1e-6):
        super().__init__()
        self.active_task = active_task
        self.unit_variances = gpflow.Parameter(
            np.full(n_units, init_var, dtype=np.float64),
            transform=gpflow.utilities.positive(lower=lower),
        )

    def __call__(self, X):
        task_ids = tf.cast(tf.round(X[..., self.active_task]), tf.int32)
        variances = tf.gather(self.unit_variances, task_ids)
        return variances[..., None]


# ---------------------------------------------------------
# Kernel: Exchangeable (shared mean + shared or per-unit X kernel)
# Cov(y_i(t), y_j(t')) = k_mu(t,t') + I(i==j) * k_x(t,t')
# If exchangeable=False, k_mu is dropped → block-diagonal
# If multiple=True, each unit has its own k_x parameters
# ---------------------------------------------------------
class GPKernel(gpflow.kernels.Kernel):
    """
    Unified kernel supporting:
      - exchangeable=True  + multiple=False : shared k_mu + shared k_x per unit
      - exchangeable=True  + multiple=True  : shared k_mu + per-unit k_x
      - exchangeable=False + multiple=False : block-diagonal, shared k_x
      - exchangeable=False + multiple=True  : block-diagonal, per-unit k_x
    """
    def __init__(self, base_kernel_mu, unit_kernels_x,
                 exchangeable=True, task_dim=0, time_dim=1):
        super().__init__()
        self.base_kernel_mu = base_kernel_mu   # None if exchangeable=False
        self.unit_kernels_x = unit_kernels_x   # list of kernels (1 if shared, N if multiple)
        self.exchangeable   = exchangeable
        self.task_dim       = task_dim
        self.time_dim       = time_dim

    def K(self, X, X2=None):
        if X2 is None:
            X2 = X

        T1 = X[:,  self.time_dim:self.time_dim + 1]
        T2 = X2[:, self.time_dim:self.time_dim + 1]
        U1 = X[:,  self.task_dim]
        U2 = X2[:, self.task_dim]

        n1 = tf.shape(X)[0]
        n2 = tf.shape(X2)[0]
        K_total = tf.zeros((n1, n2), dtype=X.dtype)

        if self.exchangeable and self.base_kernel_mu is not None:
            K_total = K_total + self.base_kernel_mu.K(T1, T2)

        num_kernels = len(self.unit_kernels_x)
        for uid, k_xi in enumerate(self.unit_kernels_x):
            if num_kernels == 1:
                # shared: apply to all units (only on matching pairs)
                task_match = tf.cast(tf.equal(U1[:, None], U2[None, :]), dtype=X.dtype)
            else:
                # per-unit: apply only to unit uid
                mask_i = tf.cast(tf.equal(U1, float(uid)), dtype=X.dtype)
                mask_j = tf.cast(tf.equal(U2, float(uid)), dtype=X.dtype)
                task_match = tf.linalg.matmul(mask_i[:, None], mask_j[None, :])

            K_total = K_total + task_match * k_xi.K(T1, T2)

        return K_total

    def K_diag(self, X):
        T = X[:, self.time_dim:self.time_dim + 1]
        U = X[:, self.task_dim]

        K_diag_total = tf.zeros(tf.shape(X)[0:1], dtype=X.dtype)

        if self.exchangeable and self.base_kernel_mu is not None:
            K_diag_total = K_diag_total + self.base_kernel_mu.K_diag(T)

        num_kernels = len(self.unit_kernels_x)
        for uid, k_xi in enumerate(self.unit_kernels_x):
            if num_kernels == 1:
                K_diag_total = K_diag_total + k_xi.K_diag(T)
            else:
                mask_i = tf.cast(tf.equal(U, float(uid)), dtype=T.dtype)
                K_diag_total = K_diag_total + mask_i * k_xi.K_diag(T)

        return K_diag_total


# ---------------------------------------------------------
# Single unified fitting function
# ---------------------------------------------------------
def fit_gp(df,
           exchangeable=True,
           multiple=False,
           kernel_type="rbf",
           init_variance=1.0,
           init_lengthscale=15.0,
           init_noise=0.05,
           verbose=True):
    """
    Fit a GP model with a unified kernel.

    Parameters
    ----------
    exchangeable : bool
        If True, includes shared mean GP k_mu (exchangeable structure).
        If False, block-diagonal kernel only (independent GP structure).
    multiple : bool
        If True, each unit gets its own k_x kernel parameters.
        If False, all units share the same k_x kernel parameters.
    kernel_type : str
        'rbf' or 'brownian'.
    """

    X = df[["unit_id", "time"]].to_numpy(dtype=np.float64)
    Y = df[["y"]].to_numpy(dtype=np.float64)
    unique_units = np.sort(np.unique(X[:, 0]).astype(int))
    num_units    = len(unique_units)

    def make_kernel(variance, lengthscale):
        if kernel_type == "rbf":
            return gpflow.kernels.RBF(variance=variance, lengthscales=lengthscale)
        elif kernel_type == "brownian":
            return BrownianMotion(variance=variance)
        else:
            raise ValueError("kernel_type must be 'rbf' or 'brownian'")

    # Shared mean kernel (only used if exchangeable=True)
    base_kernel_mu = make_kernel(init_variance, init_lengthscale) if exchangeable else None

    # Unit kernels
    if multiple:
        unit_kernels_x = [make_kernel(init_variance, init_lengthscale) for _ in range(num_units)]
    else:
        unit_kernels_x = [make_kernel(init_variance, init_lengthscale)]  # single shared kernel

    kernel = GPKernel(
        base_kernel_mu=base_kernel_mu,
        unit_kernels_x=unit_kernels_x,
        exchangeable=exchangeable,
        task_dim=0,
        time_dim=1,
    )

    likelihood = gpflow.likelihoods.Gaussian(variance=init_noise)

    m = gpflow.models.GPR(data=(X, Y), kernel=kernel, likelihood=likelihood)

    opt      = gpflow.optimizers.Scipy()
    opt_logs = opt.minimize(m.training_loss, variables=m.trainable_variables)

    if verbose:
        model_label = (
            f"{'Exchangeable' if exchangeable else 'Independent'} | "
            f"{'Multiple' if multiple else 'Shared'} | "
            f"{kernel_type.upper()}"
        )
        print("=" * 60)
        print(model_label)
        print("=" * 60)
        print(f"  Optimization success : {opt_logs.success}")
        print("-" * 60)

        if exchangeable:
            kmu = base_kernel_mu
            print(f"  Shared mean variance sigma2_mu : {float(kmu.variance.numpy()):.4f}")
            if kernel_type == "rbf":
                print(f"  Shared mean lengthscale ell_mu : {float(np.asarray(kmu.lengthscales.numpy()).squeeze()):.4f}")

        print(f"  Unit kernel(s):")
        for i, k_xi in enumerate(unit_kernels_x):
            label = f"Unit {unique_units[i]}" if multiple else "Shared (all units)"
            sigma2_xi = float(k_xi.variance.numpy())
            if exchangeable:
                sigma2_mu = float(base_kernel_mu.variance.numpy())
                icc = sigma2_mu / (sigma2_mu + sigma2_xi)
                icc_str = f", ICC = {icc:.4f}"
            else:
                icc_str = ""
            if kernel_type == "rbf":
                ell_xi = float(np.asarray(k_xi.lengthscales.numpy()).squeeze())
                print(f"    {label}: sigma2_x = {sigma2_xi:.4f}, ell_x = {ell_xi:.4f}{icc_str}")
            else:
                print(f"    {label}: sigma2_x = {sigma2_xi:.4f} (Brownian){icc_str}")

        print(f"  Noise variance sigma2_eps : {float(m.likelihood.variance.numpy()):.5f}")
        print("=" * 60)

    return m

C:\Users\barbounakis\gp_env\lib\site-packages\gpflow\versions.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [7]:
import time

df_s1 = all_simulations["sim_0"]
df_s1 = df_s1[df_s1["scenario"] == "1_exchangeable"].copy()

models = {
    "exch_shared_rbf"    : dict(exchangeable=True,  multiple=False, kernel_type="rbf"),
    "exch_shared_bm"     : dict(exchangeable=True,  multiple=False, kernel_type="brownian"),
    "exch_multiple_rbf"  : dict(exchangeable=True,  multiple=True,  kernel_type="rbf"),
    "exch_multiple_bm"   : dict(exchangeable=True,  multiple=True,  kernel_type="brownian"),
    "indep_shared_rbf"   : dict(exchangeable=False, multiple=False, kernel_type="rbf"),
    "indep_shared_bm"    : dict(exchangeable=False, multiple=False, kernel_type="brownian"),
    "indep_multiple_rbf" : dict(exchangeable=False, multiple=True,  kernel_type="rbf"),
    "indep_multiple_bm"  : dict(exchangeable=False, multiple=True,  kernel_type="brownian"),
}

fitted   = {}
timings  = {}

for name, kwargs in models.items():
    t0 = time.perf_counter()
    fitted[name] = fit_gp(df_s1, **kwargs, verbose=True)
    timings[name] = time.perf_counter() - t0

# ── Print timing summary ───────────────────────────────────
print("\n" + "=" * 45)
print(f"{'Model':<25}  {'Time (s)':>10}")
print("=" * 45)
for name, elapsed in timings.items():
    print(f"{name:<25}  {elapsed:>10.3f}s")
print("=" * 45)
print(f"{'TOTAL':<25}  {sum(timings.values()):>10.3f}s")
print("=" * 45)

Exchangeable | Shared | RBF
  Optimization success : True
------------------------------------------------------------
  Shared mean variance sigma2_mu : 5.3682
  Shared mean lengthscale ell_mu : 23.7478
  Unit kernel(s):
    Shared (all units): sigma2_x = 0.9558, ell_x = 9.8479, ICC = 0.1511
  Noise variance sigma2_eps : 1.00681
Exchangeable | Shared | BROWNIAN
  Optimization success : True
------------------------------------------------------------
  Shared mean variance sigma2_mu : 0.1050
  Unit kernel(s):
    Shared (all units): sigma2_x = 0.0948 (Brownian), ICC = 0.4747
  Noise variance sigma2_eps : 0.92371
Exchangeable | Multiple | RBF
  Optimization success : True
------------------------------------------------------------
  Shared mean variance sigma2_mu : 5.9820
  Shared mean lengthscale ell_mu : 22.9267
  Unit kernel(s):
    Unit 0: sigma2_x = 0.7644, ell_x = 9.4556, ICC = 0.1133
    Unit 1: sigma2_x = 1.2977, ell_x = 10.5022, ICC = 0.1783
    Unit 2: sigma2_x = 2.3001, ell

In [8]:
# =========================================================
# Prequential GP evaluation — CORRECTED
#
# Changes vs original:
#   1. Pointwise scoring uses predict_y (latent var + noise),
#      instead of predict_f. Means are unchanged → MAE/RMSE
#      are identical; only variance-based scores change.
#   2. Joint scoring keeps predict_f(full_cov=True) for the
#      latent covariance, then adds sigma2_eps * I to the
#      DIAGONAL by hand (predict_y(full_cov=True) is buggy in
#      GPflow — it adds noise to every element, not just the
#      diagonal; see GPflow issue #1461). This both corrects
#      the score and regularises the near-singular matrices
#      that caused the multiple-kernel blow-ups.
#   3. multivariate_log_score now REPORTS when it falls back
#      to the marginal score (Sigma not PD) instead of doing
#      it silently. A 'fallback' flag is propagated.
#   4. Aggregation expanded:
#        - per-sim then across-sim summaries
#        - mean, median, sd, var, 95% percentile interval,
#          95% CI on the mean
#        - aggregate SUM (total prequential log score) per sim,
#          then summarised across sims (mean & median of totals)
#        - everything computed for ALL runs and for
#          SUCCESSFULLY-OPTIMISED runs separately
# =========================================================

import numpy as np
import pandas as pd
import tensorflow as tf
import gpflow
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)


# =========================================================
# Brownian motion kernel: K(t,t') = sigma2 * min(t,t')
# =========================================================
class BrownianMotion(gpflow.kernels.Kernel):
    def __init__(self, variance=1.0):
        super().__init__()
        self.variance = gpflow.Parameter(variance, transform=gpflow.utilities.positive())

    def K(self, X, X2=None):
        if X2 is None:
            X2 = X
        t1 = X[:, 0:1]
        t2 = X2[:, 0:1]
        return self.variance * tf.minimum(t1, tf.transpose(t2))

    def K_diag(self, X):
        return self.variance * tf.reshape(X[:, 0], (-1,))


# =========================================================
# Exchangeable kernel
# Cov(y_i(t), y_j(t')) = k_mu(t,t') + I(i==j) * k_x(t,t')
# base_kernel_mu = None  →  independent (no shared mean)
# unit_kernels_x = [k]   →  shared unit kernel across units
# unit_kernels_x = [k0, k1, ...] → per-unit kernels
# =========================================================
class ExchangeableKernel(gpflow.kernels.Kernel):
    def __init__(self, base_kernel_mu, unit_kernels_x, task_dim=0, time_dim=1):
        super().__init__()
        self.base_kernel_mu = base_kernel_mu
        self.unit_kernels_x = unit_kernels_x
        self.task_dim = task_dim
        self.time_dim = time_dim

    def K(self, X, X2=None):
        if X2 is None:
            X2 = X
        T1 = X[:,  self.time_dim:self.time_dim + 1]
        T2 = X2[:, self.time_dim:self.time_dim + 1]
        U1 = X[:,  self.task_dim]
        U2 = X2[:, self.task_dim]

        n1 = tf.shape(X)[0]
        n2 = tf.shape(X2)[0]
        K_total = tf.zeros((n1, n2), dtype=X.dtype)

        if self.base_kernel_mu is not None:
            K_total = K_total + self.base_kernel_mu.K(T1, T2)

        num_kernels = len(self.unit_kernels_x)
        for uid, k_xi in enumerate(self.unit_kernels_x):
            if num_kernels == 1:
                task_match = tf.cast(tf.equal(U1[:, None], U2[None, :]), dtype=X.dtype)
            else:
                mask_i = tf.cast(tf.equal(U1, float(uid)), dtype=X.dtype)
                mask_j = tf.cast(tf.equal(U2, float(uid)), dtype=X.dtype)
                task_match = tf.linalg.matmul(mask_i[:, None], mask_j[None, :])
            K_total = K_total + task_match * k_xi.K(T1, T2)

        return K_total

    def K_diag(self, X):
        T = X[:, self.time_dim:self.time_dim + 1]
        U = X[:, self.task_dim]

        K_diag_total = tf.zeros(tf.shape(X)[0:1], dtype=X.dtype)

        if self.base_kernel_mu is not None:
            K_diag_total = K_diag_total + self.base_kernel_mu.K_diag(T)

        num_kernels = len(self.unit_kernels_x)
        for uid, k_xi in enumerate(self.unit_kernels_x):
            if num_kernels == 1:
                K_diag_total = K_diag_total + k_xi.K_diag(T)
            else:
                mask_i = tf.cast(tf.equal(U, float(uid)), dtype=T.dtype)
                K_diag_total = K_diag_total + mask_i * k_xi.K_diag(T)

        return K_diag_total


# =========================================================
# Model specs: 4 models per kernel
# =========================================================
def get_model_specs(kernel_type):
    return [
        {"model_name": f"exch_shared_{kernel_type}",   "exchangeable": True,  "multiple": False},
        {"model_name": f"exch_multiple_{kernel_type}", "exchangeable": True,  "multiple": True},
        {"model_name": f"indep_shared_{kernel_type}",  "exchangeable": False, "multiple": False},
        {"model_name": f"indep_multiple_{kernel_type}","exchangeable": False, "multiple": True},
    ]


# =========================================================
# Unified fit function
# =========================================================
def fit_gp(df,
           exchangeable=True,
           multiple=False,
           kernel_type="rbf",
           init_variance=1.0,
           init_lengthscale=15.0,
           init_noise=0.05,
           maxiter=200,
           verbose=False):

    X = df[["unit_id", "time"]].to_numpy(dtype=np.float64)
    Y = df[["y"]].to_numpy(dtype=np.float64)
    num_units = int(len(np.unique(X[:, 0])))

    def make_kernel():
        if kernel_type == "rbf":
            return gpflow.kernels.RBF(variance=init_variance, lengthscales=init_lengthscale)
        elif kernel_type == "brownian":
            return BrownianMotion(variance=init_variance)
        else:
            raise ValueError("kernel_type must be 'rbf' or 'brownian'")

    base_kernel_mu = make_kernel() if exchangeable else None
    num_x_kernels  = num_units if multiple else 1
    unit_kernels_x = [make_kernel() for _ in range(num_x_kernels)]

    kernel = ExchangeableKernel(
        base_kernel_mu=base_kernel_mu,
        unit_kernels_x=unit_kernels_x,
        task_dim=0, time_dim=1,
    )

    likelihood = gpflow.likelihoods.Gaussian(variance=init_noise)
    model      = gpflow.models.GPR(data=(X, Y), kernel=kernel, likelihood=likelihood)

    opt      = gpflow.optimizers.Scipy()
    opt_logs = opt.minimize(
        model.training_loss,
        variables=model.trainable_variables,
        options={"maxiter": maxiter},
        compile=False,
    )

    if verbose:
        label = (f"{'Exchangeable' if exchangeable else 'Independent'} | "
                 f"{'Multiple' if multiple else 'Shared'} | {kernel_type.upper()}")
        print(f"  [{label}] success={opt_logs.success}")

    return model, opt_logs


# =========================================================
# Pointwise predict — CORRECTED to predict_y
# predict_y returns mean (== predict_f mean) and
# var = var_f + sigma2_eps  (marginal, noise included).
# =========================================================
def predict_gp(model, df_test):
    Xnew = df_test[["unit_id", "time"]].to_numpy(dtype=np.float64)
    mean, var = model.predict_y(Xnew)          # <-- was predict_f
    out = df_test.copy()
    out["pred_mean"]  = mean.numpy().flatten()
    out["pred_var"]   = var.numpy().flatten()
    out["pred_sd"]    = np.sqrt(np.maximum(out["pred_var"], 1e-12))
    out["pred_lo_95"] = out["pred_mean"] - 1.96 * out["pred_sd"]
    out["pred_hi_95"] = out["pred_mean"] + 1.96 * out["pred_sd"]
    return out


# =========================================================
# Hyperparameter extraction (unchanged)
# =========================================================
def extract_hyperparams(model, simulation, model_name, origin,
                        exchangeable, multiple, kernel_type,
                        optim_success, fit_time):
    kernel = model.kernel
    noise  = float(np.asarray(model.likelihood.variance.numpy()).squeeze())

    if exchangeable:
        sigma2_mu = float(kernel.base_kernel_mu.variance.numpy())
        ls_mu     = (float(np.asarray(kernel.base_kernel_mu.lengthscales.numpy()).squeeze())
                     if kernel_type == "rbf" else np.nan)
    else:
        sigma2_mu = np.nan
        ls_mu     = np.nan

    rows = []
    for uid, k_xi in enumerate(kernel.unit_kernels_x):
        sigma2_xi = float(k_xi.variance.numpy())
        ls_xi     = (float(np.asarray(k_xi.lengthscales.numpy()).squeeze())
                     if kernel_type == "rbf" else np.nan)
        icc_i = (sigma2_mu / (sigma2_mu + sigma2_xi)
                 if (exchangeable and not np.isnan(sigma2_mu) and (sigma2_mu + sigma2_xi) > 0)
                 else np.nan)
        rows.append({
            "simulation": simulation, "model_name": model_name, "origin": origin,
            "unit_kernel_id": uid if multiple else "shared",
            "sigma2_mu": sigma2_mu, "ls_mu": ls_mu,
            "sigma2_x": sigma2_xi, "ls_x": ls_xi, "icc": icc_i, "noise": noise,
            "optim_success": int(optim_success), "fit_time": fit_time,
        })
    return rows


# =========================================================
# Pointwise scoring helpers
# =========================================================
def gaussian_log_score(y, mu, var):
    var = np.maximum(var, 1e-12)
    return 0.5 * np.log(2.0 * np.pi * var) + 0.5 * ((y - mu) ** 2) / var


def compute_pointwise_scores(df_pred):
    out = df_pred.copy()
    out["error"]     = out["y"] - out["pred_mean"]
    out["abs_error"] = np.abs(out["error"])
    out["sq_error"]  = out["error"] ** 2
    out["log_score"] = gaussian_log_score(out["y"].values, out["pred_mean"].values, out["pred_var"].values)
    return out


# =========================================================
# Multivariate log score — now returns (score, used_fallback)
# =========================================================
def multivariate_log_score(y_vec, mu_vec, Sigma):
    """
    Negative log density of MVN:
      0.5 * (n log(2π) + log det Σ + (y-μ)^T Σ^{-1} (y-μ))
    Returns (score, used_fallback). used_fallback=True means Σ was
    not positive-definite and the marginal score was used instead.
    """
    y     = np.asarray(y_vec).reshape(-1, 1)
    mu    = np.asarray(mu_vec).reshape(-1, 1)
    n     = y.shape[0]
    Sigma = np.asarray(Sigma, dtype=np.float64)

    # Positive-definiteness / conditioning test on the NATIVE matrix
    # (before any jitter), so the test reflects the matrix's true
    # conditioning rather than the jitter floor. We reject matrices that
    # are non-PD OR severely ill-conditioned: there Sigma^{-1} explodes and
    # the quadratic form blows up — the mechanism behind the multiple-kernel
    # blow-ups seen in the results. Eigenvalues give an unambiguous
    # condition number.
    COND_MAX = 1e10  # reject if lambda_max / lambda_min exceeds this
    use_fallback = False
    try:
        evals = np.linalg.eigvalsh(Sigma)          # symmetric -> real eigenvalues
        lmin, lmax = float(evals[0]), float(evals[-1])
        cond = (lmax / lmin) if lmin > 0 else np.inf
        if (not np.isfinite(lmin)) or (lmin <= 0) or (cond > COND_MAX):
            use_fallback = True
        else:
            # scale-aware jitter (tiny relative to the matrix) for the factorisation
            jitter = 1e-10 * max(lmax, 1.0)
            L = np.linalg.cholesky(Sigma + np.eye(n) * jitter)
    except np.linalg.LinAlgError:
        use_fallback = True

    if use_fallback:
        Sigma = Sigma + np.eye(n) * 1e-8  # stabilise the marginal fallback only
        diff     = y.flatten() - mu.flatten()
        marg_var = np.maximum(np.diag(Sigma), 1e-12)
        score = 0.5 * np.sum(np.log(2.0 * np.pi * marg_var) + (diff**2) / marg_var)
        return float(score), True

    # logdet = 2 * sum(log(diag(L)));  quad = ||L^{-1} (y-mu)||^2 via solve
    diff   = y - mu
    logdet = 2.0 * float(np.sum(np.log(np.diag(L))))
    z      = np.linalg.solve(L, diff)
    quad   = float((z.T @ z).item())
    score  = 0.5 * (n * np.log(2.0 * np.pi) + logdet + quad)
    return float(score), False


# =========================================================
# Joint score per time step — CORRECTED
# Uses latent full covariance from predict_f, then adds
# sigma2_eps * I to the diagonal (observation-level joint cov).
# =========================================================
def joint_score_per_timestep(model, test_df):
    noise = float(np.asarray(model.likelihood.variance.numpy()).squeeze())
    rows = []
    for t in sorted(test_df["time"].unique()):
        df_t = test_df[test_df["time"] == t].sort_values("unit_id")
        y_t  = df_t["y"].to_numpy(dtype=np.float64)
        Xnew = df_t[["unit_id", "time"]].to_numpy(dtype=np.float64)

        mean_t, cov_t = model.predict_f(Xnew, full_cov=True)   # latent, noise-free
        mu_t    = mean_t.numpy().flatten()
        Sigma_t = cov_t.numpy()[0, :, :]
        # add observation noise on the diagonal only (units conditionally
        # independent given the latent functions) -> covariance of y
        Sigma_t = Sigma_t + noise * np.eye(Sigma_t.shape[0])

        score_t, used_fallback = multivariate_log_score(y_t, mu_t, Sigma_t)
        rows.append({
            "time": t,
            "n_units": len(y_t),
            "joint_log_score": score_t,
            "fallback": int(used_fallback),
        })
    return rows


# =========================================================
# Optimisation success helper
# =========================================================
def scipy_result_success(res):
    if res is None:
        return False
    if hasattr(res, "success"):
        return bool(res.success)
    if isinstance(res, dict) and "success" in res:
        return bool(res["success"])
    return True


# =========================================================
# Tidy aggregation helpers
# =========================================================
def _across_sim_block(series):
    """Summarise a per-simulation statistic across simulations."""
    s = pd.Series(series).dropna()
    n = int(s.size)
    mean = float(s.mean()) if n else np.nan
    sd   = float(s.std(ddof=1)) if n > 1 else np.nan
    se   = sd / np.sqrt(n) if (n > 1 and np.isfinite(sd)) else np.nan
    return {
        "n_sims":    n,
        "mean":      mean,
        "median":    float(s.median()) if n else np.nan,
        "sd":        sd,
        "var":       float(s.var(ddof=1)) if n > 1 else np.nan,
        "q2_5":      float(s.quantile(0.025)) if n else np.nan,
        "q97_5":     float(s.quantile(0.975)) if n else np.nan,
        "ci95_lo":   mean - 1.96 * se if np.isfinite(se) else np.nan,
        "ci95_hi":   mean + 1.96 * se if np.isfinite(se) else np.nan,
        "grand_sum": float(s.sum()) if n else np.nan,
    }


def _per_sim_metric(d, metric, value_col):
    """One value per (simulation, model_name) for a given metric."""
    grp = d.groupby(["simulation", "model_name"])[value_col]
    if metric == "rmse":                      # value_col = sq_error
        out = np.sqrt(grp.mean())
    elif metric == "sum":                     # total / additive score
        out = grp.sum()
    else:                                     # mean of the column
        out = grp.mean()
    return out.reset_index(name="v")


def summarise_metric(long_df, value_col, metric, score_type, include_total):
    """
    Build tidy summary rows for one metric, across every combination of
    horizon_scope (full / first_day) x run_set (all / successful) x
    agg (per_sim_mean / per_sim_total*).  *totals only when include_total.
    """
    rows = []
    for scope in ("full", "first_day"):
        d_scope = long_df if scope == "full" else long_df[long_df["horizon_step"] == 1]
        for run_set in ("all", "successful"):
            d = d_scope if run_set == "all" else d_scope[d_scope["optim_success"] == 1]
            if d.empty:
                continue

            agg_specs = [("per_sim_mean", metric)]
            if include_total:
                agg_specs.append(("per_sim_total", "sum"))

            for agg_name, per_sim_metric in agg_specs:
                ps = _per_sim_metric(d, per_sim_metric, value_col)
                for model, sub in ps.groupby("model_name"):
                    rows.append({
                        "score_type":    score_type,
                        "metric":        metric,
                        "horizon_scope": scope,
                        "run_set":       run_set,
                        "aggregation": agg_name,
                        "model_name":    model,
                        **_across_sim_block(sub["v"]),
                    })
    return rows


def build_summarised_scores(predictions_all, joint_all):
    """
    Single tidy table summarising all scores across simulations.
    Columns: score_type, metric, horizon_scope, run_set, agg, model_name,
             n_sims, mean, median, sd, var, q2_5, q97_5, ci95_lo, ci95_hi, grand_sum
    """
    rows = []

    # joint log score (additive -> totals meaningful)
    if not joint_all.empty:
        rows += summarise_metric(joint_all, "joint_log_score",
                                 "joint_log_score", "joint", include_total=True)

    # pointwise scores
    if not predictions_all.empty:
        # log score: mean and total
        rows += summarise_metric(predictions_all, "log_score",
                                 "log_score", "pointwise", include_total=True)
        # mae: mean of abs_error
        rows += summarise_metric(predictions_all, "abs_error",
                                 "mae", "pointwise", include_total=False)
        # rmse: sqrt(mean(sq_error))
        rows += summarise_metric(predictions_all, "sq_error",
                                 "rmse", "pointwise", include_total=False)

    cols = ["score_type", "metric", "horizon_scope", "run_set", "aggregation", "model_name",
            "n_sims", "mean", "median", "sd", "var",
            "q2_5", "q97_5", "ci95_lo", "ci95_hi", "grand_sum"]
    df = pd.DataFrame(rows)
    return df[cols] if not df.empty else pd.DataFrame(columns=cols)


def build_success_rate(optim_all, joint_all):
    """Optimisation success summarised across all simulations, per model."""
    if optim_all.empty:
        return pd.DataFrame()

    pooled = (optim_all.groupby("model_name", as_index=False)
              .agg(n_runs=("optim_success", "size"),
                   n_success=("optim_success", "sum"))
              .assign(success_rate=lambda d: d["n_success"] / d["n_runs"],
                      success_pct=lambda d: 100.0 * d["n_success"] / d["n_runs"]))

    per_sim = (optim_all.groupby(["simulation", "model_name"], as_index=False)
               .agg(rate=("optim_success", "mean")))
    per_sim_summ = (per_sim.groupby("model_name", as_index=False)
                    .agg(mean_success_rate_per_sim=("rate", "mean"),
                         sd_success_rate_per_sim=("rate", "std")))

    out = pooled.merge(per_sim_summ, on="model_name", how="left")

    # scoring-health diagnostic: how often the joint covariance fell back
    if not joint_all.empty and "fallback" in joint_all.columns:
        fb = (joint_all.groupby("model_name", as_index=False)
              .agg(n_joint_timesteps=("fallback", "size"),
                   n_fallback=("fallback", "sum"))
              .assign(fallback_pct=lambda d: 100.0 * d["n_fallback"] / d["n_joint_timesteps"]))
        out = out.merge(fb, on="model_name", how="left")

    return out.sort_values("model_name").reset_index(drop=True)


# =========================================================
# Prequential: one simulation   (step == horizon, non-overlapping)
# =========================================================
def prequential_one_simulation(df, simulation_name="sim", kernel_type="rbf",
                               train_end_start=50, horizon=7,
                               maxiter=200, verbose=True):
    model_specs = get_model_specs(kernel_type)
    df       = df.copy().sort_values(["unit_id", "time"]).reset_index(drop=True)
    max_time = df["time"].max()

    # step is fixed to horizon -> non-overlapping windows, each point scored once
    step = horizon
    origins   = []
    train_end = train_end_start
    while train_end + horizon <= max_time:
        origins.append(train_end)
        train_end += step

    if verbose:
        print(f"  Simulation {simulation_name} | kernel={kernel_type} | "
              f"horizon={horizon} | origins={origins}")

    prediction_rows, joint_rows = [], []
    hyperparam_rows, timing_rows, optim_rows = [], [], []

    for spec in model_specs:
        model_name   = spec["model_name"]
        exchangeable = spec["exchangeable"]
        multiple     = spec["multiple"]
        if verbose:
            print(f"    Model: {model_name}")

        for origin in origins:
            train_df = df[df["time"] <= origin].copy()
            test_df  = df[(df["time"] > origin) & (df["time"] <= origin + horizon)].copy()
            if test_df.empty:
                continue
            try:
                t0 = time.perf_counter()
                model, opt_res = fit_gp(train_df, exchangeable=exchangeable,
                                        multiple=multiple, kernel_type=kernel_type,
                                        maxiter=maxiter, verbose=False)
                fit_time      = time.perf_counter() - t0
                optim_success = scipy_result_success(opt_res)

                # pointwise
                pred_df = predict_gp(model, test_df)
                pred_df = compute_pointwise_scores(pred_df)
                pred_df["simulation"]    = simulation_name
                pred_df["model_name"]    = model_name
                pred_df["origin"]        = origin
                pred_df["horizon_step"]  = pred_df["time"] - origin
                pred_df["first_day"]     = (pred_df["horizon_step"] == 1).astype(int)
                pred_df["optim_success"] = int(optim_success)
                pred_df["fit_time"]      = fit_time
                prediction_rows.append(pred_df)

                # joint (per timestep across units)
                for ts in joint_score_per_timestep(model, test_df):
                    hstep = ts["time"] - origin
                    joint_rows.append({
                        "simulation": simulation_name, "model_name": model_name,
                        "origin": origin, "time": ts["time"], "horizon_step": hstep,
                        "first_day": int(hstep == 1), "n_units": ts["n_units"],
                        "joint_log_score": ts["joint_log_score"], "fallback": ts["fallback"],
                        "optim_success": int(optim_success), "fit_time": fit_time,
                    })

                # hyperparameters
                hyperparam_rows.extend(extract_hyperparams(
                    model, simulation_name, model_name, origin,
                    exchangeable=exchangeable, multiple=multiple,
                    kernel_type=kernel_type, optim_success=optim_success,
                    fit_time=fit_time))

                timing_rows.append({"simulation": simulation_name, "model_name": model_name,
                                    "origin": origin, "fit_time": fit_time})
                optim_rows.append({"simulation": simulation_name, "model_name": model_name,
                                   "origin": origin, "optim_success": int(optim_success),
                                   "fit_time": fit_time})

            except Exception as e:
                if verbose:
                    print(f"      Failed at origin {origin}: {e}")
                timing_rows.append({"simulation": simulation_name, "model_name": model_name,
                                    "origin": origin, "fit_time": np.nan})
                optim_rows.append({"simulation": simulation_name, "model_name": model_name,
                                   "origin": origin, "optim_success": 0, "fit_time": np.nan})

    def _cat(lst, df_fallback=pd.DataFrame()):
        return pd.concat(lst, axis=0).reset_index(drop=True) if lst else df_fallback

    return (_cat(prediction_rows),
            pd.DataFrame(joint_rows),
            pd.DataFrame(hyperparam_rows),
            pd.DataFrame(timing_rows),
            pd.DataFrame(optim_rows))


# =========================================================
# Prequential: all simulations  ->  5 tidy outputs
# =========================================================
def prequential_all_simulations(all_simulations, kernel_type="rbf", scenario=None,
                                simulation_keys=None, train_end_start=50,
                                horizon=7, maxiter=200, verbose=True):
    """
    Returns a dict with five tidy tables (each becomes one Excel sheet):

      model_parameters  : fitted kernel hyperparameters per (sim, model, origin, unit)
      predictions       : predicted values per observation (+ truth, first_day, optim_success)
      individual_scores : per-observation scores + optim_success (+ first_day)
      summarised_scores : scores summarised across simulations (mean/median/sd/var/95%/total),
                          for full horizon AND first-day, ALL AND successful runs
      success_rate      : optimisation success summarised across all simulations, per model

    Also returns (in memory, not written by default):
      joint_individual  : raw per-timestep joint log scores (feed the joint summary)
      meta              : run metadata
    """
    if simulation_keys is None:
        simulation_keys = sorted(all_simulations.keys())

    all_pred, all_joint, all_hyper, all_time, all_optim = [], [], [], [], []
    total_start = time.perf_counter()

    for sim_key in simulation_keys:
        if verbose:
            print("=" * 70)
            print(f"Simulation: {sim_key} | kernel: {kernel_type}")
            print("=" * 70)
        df = all_simulations[sim_key].copy()
        if scenario is not None and "scenario" in df.columns:
            df = df[df["scenario"] == scenario].copy()

        preds, joint, hps, times, opts = prequential_one_simulation(
            df=df, simulation_name=sim_key, kernel_type=kernel_type,
            train_end_start=train_end_start, horizon=horizon,
            maxiter=maxiter, verbose=verbose)

        all_pred.append(preds); all_joint.append(joint); all_hyper.append(hps)
        all_time.append(times); all_optim.append(opts)

    total_elapsed = time.perf_counter() - total_start

    def _cat(lst):
        lst = [x for x in lst if x is not None and not x.empty]
        return pd.concat(lst, axis=0).reset_index(drop=True) if lst else pd.DataFrame()

    predictions_all = _cat(all_pred)
    joint_all       = _cat(all_joint)
    hyperparams_all = _cat(all_hyper)
    optim_all       = _cat(all_optim)

    # ── Sheet 1: model parameters (fitted hyperparameters) ──
    model_parameters = hyperparams_all

    # ── Sheet 2: predictions (predicted values only) ──
    pred_cols = ["simulation", "model_name", "origin", "unit_id", "time",
                 "horizon_step", "first_day",
                 "y", "true_x", "true_mu",
                 "pred_mean", "pred_var", "pred_sd", "pred_lo_95", "pred_hi_95",
                 "optim_success"]
    predictions = (predictions_all[[c for c in pred_cols if c in predictions_all.columns]]
                   if not predictions_all.empty else pd.DataFrame(columns=pred_cols))

    # ── Sheet 3: individual scores (per observation) + optim_success ──
    score_cols = ["simulation", "model_name", "origin", "unit_id", "time",
                  "horizon_step", "first_day",
                  "error", "abs_error", "sq_error",
                  "log_score",
                  "optim_success"]
    individual_scores = (predictions_all[[c for c in score_cols if c in predictions_all.columns]]
                         if not predictions_all.empty else pd.DataFrame(columns=score_cols))

    # ── Sheet 4: summarised scores ──
    summarised_scores = build_summarised_scores(predictions_all, joint_all)

    # ── Sheet 5: success rate ──
    success_rate = build_success_rate(optim_all, joint_all)

    if verbose:
        print(f"\n{'='*70}\nTOTAL WALL TIME: {total_elapsed:.1f}s\n{'='*70}")
        if not summarised_scores.empty:
            head = summarised_scores[
                (summarised_scores.score_type == "joint") &
                (summarised_scores.horizon_scope == "full") &
                (summarised_scores.run_set == "all") &
                (summarised_scores["aggregation"] == "per_sim_mean")]
            print("\n── Joint log score | full horizon | ALL runs | per-sim mean ──")
            print(head[["model_name", "mean", "median", "sd", "q2_5", "q97_5"]]
                  .to_string(index=False))
        if not success_rate.empty:
            print("\n── Optimisation success across all simulations ──")
            cols = [c for c in ["model_name", "n_runs", "success_pct",
                                "n_fallback", "fallback_pct"] if c in success_rate.columns]
            print(success_rate[cols].to_string(index=False))

    return {
        "model_parameters":  model_parameters,
        "predictions":       predictions,
        "individual_scores": individual_scores,
        "summarised_scores": summarised_scores,
        "success_rate":      success_rate,
        # extras (in memory; not written to the 5-sheet workbook by default)
        "joint_individual":  joint_all,
        "meta": {"kernel_type": kernel_type, "scenario": scenario,
                 "horizon": horizon, "train_end_start": train_end_start,
                 "n_simulations": len(simulation_keys),
                 "total_wall_time_seconds": total_elapsed},
    }


# =========================================================
# Write the five tidy tables to one Excel workbook
# =========================================================
def write_results_to_excel(result, path, include_joint_individual=False):
    """Write the 5 core sheets (optionally a 6th raw joint sheet) to `path`."""
    sheets = ["model_parameters", "predictions", "individual_scores",
              "summarised_scores", "success_rate"]
    if include_joint_individual:
        sheets.append("joint_individual")
    with pd.ExcelWriter(path, engine="openpyxl") as xl:
        for name in sheets:
            df = result.get(name, pd.DataFrame())
            (df if isinstance(df, pd.DataFrame) else pd.DataFrame()).to_excel(
                xl, sheet_name=name[:31], index=False)
    return path


# =========================================================
# Example usage
# =========================================================
# res = prequential_all_simulations(
#     all_simulations=all_simulations, kernel_type="rbf",
#     scenario="1_exchangeable", train_end_start=50, horizon=7, verbose=True)
#
# res["summarised_scores"]   # tidy: filter by score_type / horizon_scope / run_set / agg
# res["success_rate"]        # optimisation success + fallback diagnostics per model
#
# # one-day-ahead joint log score, successful runs only:
# s = res["summarised_scores"]
# s[(s.score_type=="joint") & (s.horizon_scope=="first_day") &
#   (s.run_set=="successful") & (s["aggregation"]=="per_sim_mean")]
#
# write_results_to_excel(res, "results_1_exchangeable_horizon_7_rbf.xlsx")

In [9]:
scenarios = ["1_exchangeable", "2_nearly_independent", "3_non_exchangeable_iid", "4_non_exchangeable_non_iid"]
for sc in scenarios:
    res = prequential_all_simulations(
        all_simulations, kernel_type="rbf", scenario=sc,
        train_end_start=56, horizon=7, verbose=True,
    )
    write_results_to_excel(res, f"results_{sc}_horizon_7_rbf.xlsx")

Simulation: sim_0 | kernel: rbf
  Simulation sim_0 | kernel=rbf | horizon=7 | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_1 | kernel: rbf
  Simulation sim_1 | kernel=rbf | horizon=7 | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_10 | kernel: rbf
  Simulation sim_10 | kernel=rbf | horizon=7 | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_11 | kernel: rbf
  Simulation sim_11 | kernel=rbf | horizon=7 | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_12 | kernel: rbf
  Simulation sim_12 | kerne

In [10]:
scenarios = ["1_exchangeable", "2_nearly_independent", "3_non_exchangeable_iid", "4_non_exchangeable_non_iid"]
for sc in scenarios:
    res = prequential_all_simulations(
        all_simulations, kernel_type="brownian", scenario=sc,
        train_end_start=56, horizon=7, verbose=True,
    )
    write_results_to_excel(res, f"results_{sc}_horizon_7_brownian.xlsx")

Simulation: sim_0 | kernel: brownian
  Simulation sim_0 | kernel=brownian | horizon=7 | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_brownian
    Model: exch_multiple_brownian
    Model: indep_shared_brownian
    Model: indep_multiple_brownian
Simulation: sim_1 | kernel: brownian
  Simulation sim_1 | kernel=brownian | horizon=7 | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_brownian
    Model: exch_multiple_brownian
    Model: indep_shared_brownian
    Model: indep_multiple_brownian
Simulation: sim_10 | kernel: brownian
  Simulation sim_10 | kernel=brownian | horizon=7 | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_brownian
    Model: exch_multiple_brownian
    Model: indep_shared_brownian
    Model: indep_multiple_brownian
Simulation: sim_11 | kernel: brownian
  Simulation sim_11 | kernel=brownian | horizon=7 | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_brownian
    Model: exch_multiple_brownian
    Mode

In [8]:
results_h1_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="1_exchangeable",
    train_end_start=56,
    horizon=1,
    step=1,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_1_exchangeable_horizon_1_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results_h1_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")


results_h7_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="1_exchangeable",
    train_end_start=56,
    horizon=7,
    step=7,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_1_exchangeable_horizon_7_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results_h7_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")


results_h14_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="1_exchangeable",
    train_end_start=56,
    horizon=14,
    step=14,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_1_exchangeable_horizon_14_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results_h14_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")


Simulation: sim_0 | kernel: rbf
  Simulation sim_0 | kernel=rbf | origins=[56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_1 | kernel: rbf
  Simulation sim_1 | kernel=rbf | origins=[56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_10 | kernel: rbf
  Simulation sim_10 | kernel=rbf | origins=[56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 7

In [ ]:
results2_h1_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="2_nearly_independent",
    train_end_start=56,
    horizon=1,
    step=1,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_2_nearly_independent_horizon_1_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results2_h1_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")


In [10]:

results2_h7_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="2_nearly_independent",
    train_end_start=56,
    horizon=7,
    step=7,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_2_nearly_independent_horizon_7_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results2_h7_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")


results2_h14_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="2_nearly_independent",
    train_end_start=56,
    horizon=14,
    step=14,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_2_nearly_independent_horizon_14_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results2_h14_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")



Simulation: sim_0 | kernel: rbf
  Simulation sim_0 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_1 | kernel: rbf
  Simulation sim_1 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_10 | kernel: rbf
  Simulation sim_10 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_11 | kernel: rbf
  Simulation sim_11 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_12 | kernel: rbf
  Simulation sim_12 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105

In [ ]:

results3_h7_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="3_non_exchangeable",
    train_end_start=56,
    horizon=7,
    step=7,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_3_non_exchangeable_horizon_7_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results3_h7_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")


results3_h14_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="3_non_exchangeable",
    train_end_start=56,
    horizon=14,
    step=14,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_3_non_exchangeable_horizon_14_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results3_h14_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")



Simulation: sim_0 | kernel: rbf
  Simulation sim_0 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_1 | kernel: rbf
  Simulation sim_1 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_10 | kernel: rbf
  Simulation sim_10 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_11 | kernel: rbf
  Simulation sim_11 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105]
    Model: exch_shared_rbf
    Model: exch_multiple_rbf
    Model: indep_shared_rbf
    Model: indep_multiple_rbf
Simulation: sim_12 | kernel: rbf
  Simulation sim_12 | kernel=rbf | origins=[56, 63, 70, 77, 84, 91, 98, 105

In [ ]:
results3_h1_rbf = prequential_all_simulations(
    all_simulations=all_simulations,
    kernel_type="rbf",
    scenario="3_non_exchangeable",
    train_end_start=56,
    horizon=1,
    step=1,
    maxiter=200,
    verbose=True,
)

excel_filename = "all_results_3_non_exchangeable_horizon_1_rbf.xlsx"

with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
    for object_name, obj in results3_h1_rbf.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_excel(writer, sheet_name=object_name[:31], index=False)

print(f"All DataFrame objects saved to {excel_filename} in separate sheets!")

